# Sync Location Curations

Syncs location curations (oxjob #1170) of `is_oa`, `license`, `pdf_url`, `version` and `landing_page_url` from the users Heroku Postgres database to a local Databricks Delta table. Runs in `walden_end2end.yaml` before `ApplyLocationCurations`, which lays them over `locations_mapped` in the view `openalex.works.locations_mapped_curated`.

**Source**: `openalex_users.public.location_curations`, a typed view over `openalex_users.public.curations` (users-api migration 116). `location_id` is the API's location id, `<native_id_namespace>:<native_id>`. A removal carries the value `'null'`; read the action.

**Target**: `openalex.works.location_curations`, with the Delta change feed on: `sync_locations.py` re-sends the locations whose curations changed.

When `is_oa` and the licence go together, users-api writes both as plain curations under the submitter (oxjob #1170 README, Decision: the is_oa <-> licence pair). Rows leave Postgres only when an account is deleted (oxjob #1436, users-api migration 091); they leave this table through `NOT MATCHED BY SOURCE`, and the decline guardrail allows for rows owned by deleted accounts and refuses the clause on any other suspicious empty or short source.

## Create target table (idempotent)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.works.location_curations (
    curation_id        STRING NOT NULL,
    location_id        STRING NOT NULL,
    property           STRING NOT NULL,
    action             STRING NOT NULL,
    value              STRING,
    user_id            STRING,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP
)
USING DELTA
CLUSTER BY (location_id)
TBLPROPERTIES (delta.enableChangeDataFeed = true)

## Sync location curations

In [ ]:
%sql
-- Empty-when-target-nonempty fails unconditionally; decline check is overridable.
-- (Both 0 on startup is legitimate and doesn't fail.)
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

-- oxjob #1436: deleting an account deletes its curations in users-api, and
-- users-api records the account in deleted_users. Rows here owned by such
-- accounts are expected to go, so they widen the allowed decline; any other
-- drop still trips the guardrail.
SET VARIABLE allowed_decline = 10 + (
  SELECT COUNT(*) FROM openalex.works.location_curations t
  WHERE t.user_id IN (SELECT user_id FROM openalex_users.public.deleted_users)
);

SET VARIABLE new_count = (SELECT COUNT(*) FROM openalex_users.public.location_curations);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.works.location_curations);

SELECT
  new_count AS new_curations,
  current_count AS current_curations,
  openalex.works.e2e_override_active('guardrails_override', :guardrails_override) AS guardrails_override;  -- job parameter OR a pre-cleared row in openalex.works.e2e_overrides

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 location curations but the target holds ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source view before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND NOT openalex.works.e2e_override_active('guardrails_override', :guardrails_override)
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source location curation count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job, or pre-clear the run in openalex.works.e2e_overrides, to bypass.'))
END;

In [ ]:
%sql
MERGE INTO openalex.works.location_curations AS target
USING (
    SELECT
        curation_id, location_id, property, action, value, user_id, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.location_curations
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
SELECT
  COUNT(*)                    AS location_curation_rows,
  COUNT(DISTINCT location_id) AS locations,
  MAX(updated_datetime)       AS last_sync
FROM openalex.works.location_curations